# COPT-MT: M1b root-only hard-graph benchmark

Mục tiêu của M1b là kiểm tra có điều kiện tín hiệu yếu từ M1: COPT chỉ từng thắng ở root của đồ thị lớn, dày. Đây **không phải** huấn luyện gate và không chạy lại M1.

Thiết kế: ER $n\in\{120,150,180\}$, $p\in\{0.6,0.7,0.8\}$, 8 graph độc lập/cell, 3 repeats, cutoff 5 giây; so sánh Structural, Random và COPT trên cùng root state. Kết quả được ghi từng dòng thẳng vào Google Drive.

> Trước khi chạy: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# 1. Kiểm tra GPU
import sys, torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật T4 GPU trước khi tiếp tục.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Clone upstream nếu runtime chưa có repo
from pathlib import Path
import subprocess
repo = Path('/content/COPT-MT')
if not repo.exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/ghugn/COPT.git', str(repo)])
else:
    print('Dùng repo hiện có:', repo)
%cd /content/COPT-MT
!git log -n 1 --oneline

In [ ]:
# 3. Cài dependencies
%pip install -q hydra-core lightning torchmetrics torch-geometric hydra-submitit-launcher rootutils networkx numba POT rich wandb yacs einops dwave-networkx dimod ogb performer-pytorch python-sat

import subprocess, sys, torch
torch_version = torch.__version__.split('+')[0]
cuda_tag = 'cu' + torch.version.cuda.replace('.', '')
wheel_index = f'https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html'
print('PyG wheel index:', wheel_index)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', f'--find-links={wheel_index}', 'torch-scatter', 'torch-sparse'])

In [ ]:
# 4. Tự tìm bundle đã upload; nếu chưa có thì yêu cầu chọn file
from google.colab import files
import zipfile
candidates = list(Path('/content').rglob('guidance_kaggle_bundle*.zip'))
if candidates:
    bundle = max(candidates, key=lambda p: p.stat().st_mtime)
else:
    %cd /content
    print('Chọn guidance_kaggle_bundle.zip (tên có hậu tố (1) cũng được):')
    uploaded = files.upload()
    assert uploaded, 'Bạn chưa chọn file ZIP nào'
    uploaded_paths = [Path('/content') / name for name in uploaded]
    zip_paths = [p for p in uploaded_paths if p.suffix.lower() == '.zip']
    assert len(zip_paths) == 1, f'Cần upload đúng một file ZIP, nhận được: {uploaded_paths}'
    bundle = zip_paths[0]
print('Đang dùng bundle:', bundle)
assert zipfile.is_zipfile(bundle), f'File không phải ZIP hợp lệ: {bundle}'
with zipfile.ZipFile(bundle) as z:
    z.extractall('/content/COPT-MT')
CHECKPOINT = '/content/COPT-MT/artifacts/checkpoints/mis_rb_small/epoch_189.ckpt'
assert Path(CHECKPOINT).exists(), CHECKPOINT
print('Checkpoint:', CHECKPOINT)

In [ ]:
# 5. Unit test solver exact
%cd /content/COPT-MT
!python -m unittest experiments.guidance.test_solver -v

In [ ]:
# 6. Smoke test ngắn: xác nhận checkpoint và COPT inference hoạt động
%cd /content/COPT-MT
!rm -rf /content/guidance_smoke_m1b
!python -u -m experiments.guidance.pilot \
    --checkpoint "{CHECKPOINT}" --device cuda \
    --sizes 40 --densities 0.7 --graphs-per-cell 1 \
    --states-per-graph 0 --repeats 1 --seconds 1 \
    --output /content/guidance_smoke_m1b
print(Path('/content/guidance_smoke_m1b/summary.json').read_text())

## Chạy M1b

Thời gian xấu nhất của phần replay khoảng 54 phút (72 graph × 3 repeats × 3 methods × 5 giây), thường ngắn hơn vì nhiều lượt chứng minh xong sớm. Không chạy lại cell này vào cùng thư mục output.

In [ ]:
# 7. Mount Drive và chạy benchmark root-only
from google.colab import drive
from datetime import datetime, timezone
drive.mount('/content/drive')
result_root = Path('/content/drive/MyDrive/COPT-MT-results')
result_root.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_utc')
RUN_OUTPUT = result_root / f'guidance_pilot_m1b_{stamp}'
print('Đang ghi trực tiếp vào:', RUN_OUTPUT)
%cd /content/COPT-MT
!python -u -m experiments.guidance.pilot \
    --checkpoint "{CHECKPOINT}" --device cuda \
    --sizes 120 150 180 \
    --densities 0.6 0.7 0.8 \
    --graphs-per-cell 8 \
    --states-per-graph 0 \
    --repeats 3 \
    --seconds 5 \
    --node-limit 2000000 \
    --seed 22345 \
    --output "{RUN_OUTPUT}"

In [ ]:
# 8. Xem summary, nén và tải kết quả; bản gốc vẫn còn trên Drive
import shutil
from google.colab import files
summary = RUN_OUTPUT / 'summary.json'
assert summary.exists(), f'Benchmark chưa kết thúc hoặc lỗi; dữ liệu đã ghi nằm tại {RUN_OUTPUT}'
print(summary.read_text())
archive = shutil.make_archive('/content/guidance_pilot_m1b', 'zip', RUN_OUTPUT)
print('Bản gốc trên Drive:', RUN_OUTPUT)
files.download(archive)